# Project: Triangular FX Arbitrage Checker

## Description
This project implements a **triangular arbitrage scanner** in the foreign exchange (FX) market.  
It works by:
1. Collecting exchange rates for multiple currency pairs.  
2. Constructing possible three-currency loops (e.g., EUR → USD → JPY → EUR).  
3. Simulating trades through each loop with a fixed starting notional (10,000 units).  
4. Accounting for **transaction costs** (fees in basis points) and **slippage** on each leg.  
5. Calculating the final amount after the cycle, net return (%), and profit or loss (PnL).  

## Goal
- Detect whether a triangular loop yields a **positive return** (profitable arbitrage) or a **negative return** (loss due to costs).  
- Provide a simple trade plan (currency sequence and conversions) for each cycle.  

## Current Results
- Using the demo snapshot data, **all detected cycles show negative returns**.  
- Losses are small (around -0.03% to -0.09%), which represent the **cost drag** from spreads, fees, and slippage.  
- No arbitrage opportunities are currently exploitable in this dataset.  

## Further Steps
- Connect to a **live FX data source** (e.g., Oanda, Alpha Vantage, or broker API).  
- Tune cost parameters to reflect real trading conditions.  
- Monitor cycles in real-time to catch **short-lived arbitrage opportunities**.  


In [1]:
print("Hello, World!")

Hello, World!


In [ ]:
from decimal import Decimal, getcontext
from itertools import permutations
from datetime import datetime

# ---------- Config ----------
CURRENCIES = ["USD", "EUR", "JPY", "GBP", "AUD", "CHF", "CAD", "SGD"]
BASE_NOTIONAL = Decimal("10000")
FEE_BPS = Decimal("1.0")       # per-leg commission in bps
SLIP_BPS = Decimal("1.0")      # per-leg slippage in bps
MAX_SPREAD_BPS = Decimal("10") # sanity filter
SHOW_TOP_N = 12
VERBOSE = False

getcontext().prec = 50

def bps(x): return Decimal(x)

def bps_mult(bps_val, worsen_up=True):
    # worsen_up=True: multiply by (1 + bps/1e4), else (1 - bps/1e4)
    frac = bps_val / Decimal(10000)
    return (Decimal(1) + frac) if worsen_up else (Decimal(1) - frac)

def apply_spread(mid, spread_bps):
    half = spread_bps / Decimal(20000)
    ask = mid * (Decimal(1) + half)
    bid = mid * (Decimal(1) - half)
    return bid, ask

# ----------- Demo quotes (mids + spreads) ----------
# Provide SOME pairs; the code will add inverses automatically.
DEMO_MIDS = {
    ("EUR","USD"): Decimal("1.1000"),
    ("USD","JPY"): Decimal("150.00"),
    ("GBP","USD"): Decimal("1.2600"),
    ("AUD","USD"): Decimal("0.6700"),
    ("USD","CHF"): Decimal("0.9050"),
    ("USD","CAD"): Decimal("1.3600"),
    ("USD","SGD"): Decimal("1.3500"),
    ("EUR","JPY"): Decimal("165.00"),
    ("GBP","JPY"): Decimal("189.00"),
    ("EUR","GBP"): Decimal("0.8730"),
    ("EUR","CHF"): Decimal("0.9950"),
    ("AUD","JPY"): Decimal("100.50"),
}
DEMO_SPREAD_BPS = {
    ("EUR","USD"): bps("1.2"),
    ("USD","JPY"): bps("1.5"),
    ("GBP","USD"): bps("1.8"),
    ("AUD","USD"): bps("2.5"),
    ("USD","CHF"): bps("2.0"),
    ("USD","CAD"): bps("2.0"),
    ("USD","SGD"): bps("3.0"),
    ("EUR","JPY"): bps("2.0"),
    ("GBP","JPY"): bps("3.0"),
    ("EUR","GBP"): bps("2.0"),
    ("EUR","CHF"): bps("2.0"),
    ("AUD","JPY"): bps("3.0"),
}

def build_quote_book(mids, spreads, max_spread_bps=Decimal("10")):
    """
    Build directed quotes (bid/ask) for all provided pairs + their inverses.
    No single-base assumption -> no KeyError.
    """
    book = {}
    for (base, quote), mid in mids.items():
        spr = spreads.get((base, quote), Decimal("2.0"))
        if spr > max_spread_bps:
            continue
        bid, ask = apply_spread(mid, spr)
        book[(base, quote)] = {"mid": mid, "spr_bps": spr, "bid": bid, "ask": ask}

        # Inverse
        inv_mid = Decimal(1) / mid
        inv_bid, inv_ask = apply_spread(inv_mid, spr)  # keep bps (approx)
        book[(quote, base)] = {"mid": inv_mid, "spr_bps": spr, "bid": inv_bid, "ask": inv_ask}
    return book

def leg_direction(a, b, book):
    """
    We hold a and want b.
    If (a,b) exists, we SELL a/b at bid to receive b.
    Else if (b,a) exists, we BUY b/a at ask using a.
    """
    if (a,b) in book:
        return "sell", book[(a,b)]["bid"]
    if (b,a) in book:
        return "buy", book[(b,a)]["ask"]
    return None, None

def exec_leg(amount_in, px, side, fee_bps, slip_bps):
    # worsen effective rate by fee+slip
    total = fee_bps + slip_bps
    if side == "buy":
        eff_px = px * bps_mult(total, worsen_up=True)   # pay more
        amount_out = amount_in / eff_px
    else:
        eff_px = px * bps_mult(total, worsen_up=False)  # receive less
        amount_out = amount_in * eff_px
    return amount_out, eff_px

def check_cycle(a,b,c, book, notional, fee_bps, slip_bps):
    amt0 = notional

    s1, p1 = leg_direction(a,b,book)
    if s1 is None: return None
    amt1, e1 = exec_leg(amt0, p1, s1, fee_bps, slip_bps)

    s2, p2 = leg_direction(b,c,book)
    if s2 is None: return None
    amt2, e2 = exec_leg(amt1, p2, s2, fee_bps, slip_bps)

    s3, p3 = leg_direction(c,a,book)
    if s3 is None: return None
    amt3, e3 = exec_leg(amt2, p3, s3, fee_bps, slip_bps)

    pnl = amt3 - amt0
    ret = (amt3 / amt0) - Decimal(1)
    return {
        "cycle": f"{a}->{b}->{c}->{a}",
        "start_ccy": a,
        "start_amt": amt0,
        "end_amt": amt3,
        "pnl": pnl,
        "ret": ret,
        "legs": [
            {"leg": f"{a}->{b}", "side": s1, "px": p1, "eff_px": e1, "in": amt0, "out": amt1},
            {"leg": f"{b}->{c}", "side": s2, "px": p2, "eff_px": e2, "in": amt1, "out": amt2},
            {"leg": f"{c}->{a}", "side": s3, "px": p3, "eff_px": e3, "in": amt2, "out": amt3},
        ]
    }

def rank_cycles(currencies, book, notional, fee_bps, slip_bps, top_n=10):
    seen = set()
    out = []
    for a,b,c in permutations(currencies, 3):
        # deduplicate rotations/reflections by anchoring on min currency symbol
        if a != min(a,b,c): 
            continue
        key = tuple(sorted([a,b,c]))
        if key in seen: 
            continue
        seen.add(key)
        res1 = check_cycle(a,b,c, book, notional, fee_bps, slip_bps)
        res2 = check_cycle(a,c,b, book, notional, fee_bps, slip_bps)  # reversed inner order
        for r in (res1, res2):
            if r is not None:
                out.append(r)
    out.sort(key=lambda x: x["ret"], reverse=True)
    return out[:top_n]

def fmt(x, n=8):
    s = f"{x:.{n}f}"
    return s.rstrip('0').rstrip('.') if '.' in s else s

def main():
    print("=== Triangular FX Arbitrage Scanner ===")
    print("Currencies:", CURRENCIES)
    print(f"Costs per leg: fee={FEE_BPS} bps, slippage={SLIP_BPS} bps")
    print("Timestamp:", datetime.utcnow().isoformat(), "UTC\n")

    # Build robust quote book (no single-base dependency)
    book = build_quote_book(DEMO_MIDS, DEMO_SPREAD_BPS, MAX_SPREAD_BPS)

    # Scan
    opps = rank_cycles(CURRENCIES, book, BASE_NOTIONAL, FEE_BPS, SLIP_BPS, SHOW_TOP_N)
    if not opps:
        print("No cycles found with net positive return under current costs.")
        return

    for i, o in enumerate(opps, 1):
        pct = o["ret"] * Decimal(100)
        print(f"[{i:02d}] {o['cycle']} | Return: {fmt(pct,6)}% | PnL: {fmt(o['pnl'],6)} {o['start_ccy']}")
        if VERBOSE:
            for lg in o["legs"]:
                print(f"   {lg['leg']:>10s} | side={lg['side']:4s} | px={fmt(lg['px'],10)} "
                      f"| eff_px={fmt(lg['eff_px'],10)} | {fmt(lg['in'],6)} -> {fmt(lg['out'],6)}")
        a,b,c,_ = o["cycle"].split("->")
        print(f"   Plan: Start {fmt(o['start_amt'])} {a} → {b} → {c} → back to {a} = {fmt(o['end_amt'])} {a}\n")

if __name__ == "__main__":
    main()

=== Triangular FX Arbitrage Scanner ===
Currencies: ['USD', 'EUR', 'JPY', 'GBP', 'AUD', 'CHF', 'CAD', 'SGD']
Costs per leg: fee=1.0 bps, slippage=1.0 bps
Timestamp: 2025-10-01T06:14:16.095731 UTC

[01] CHF->EUR->USD->CHF | Return: -0.035762% | PnL: -3.576215 CHF
   Plan: Start 10000 CHF → EUR → USD → back to CHF = 9996.42378499 CHF

[02] EUR->USD->GBP->EUR | Return: -0.083154% | PnL: -8.31543 EUR
   Plan: Start 10000 EUR → USD → GBP → back to EUR = 9991.68457048 EUR

[03] EUR->USD->JPY->EUR | Return: -0.083472% | PnL: -8.34721 EUR
   Plan: Start 10000 EUR → USD → JPY → back to EUR = 9991.65278953 EUR

[04] EUR->JPY->USD->EUR | Return: -0.083472% | PnL: -8.34721 EUR
   Plan: Start 10000 EUR → JPY → USD → back to EUR = 9991.65278953 EUR

[05] EUR->GBP->USD->EUR | Return: -0.086788% | PnL: -8.67876 EUR
   Plan: Start 10000 EUR → GBP → USD → back to EUR = 9991.3212398 EUR

[06] GBP->USD->JPY->GBP | Return: -0.091466% | PnL: -9.146596 GBP
   Plan: Start 10000 GBP → USD → JPY → back to GBP =

C:\Users\Viriya Gunawan Lim\AppData\Local\Temp\ipykernel_30200\2924875225.py:163: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print("Timestamp:", datetime.utcnow().isoformat(), "UTC\n")


# Triangular FX Arbitrage Results — Interpretation

**Parameters**  
- Costs per leg: fee = 1.0 bps, slippage = 1.0 bps  
- Starting notional per cycle: 10,000 units of the base currency  
- Timestamp: 2025-10-01 06:14:16 UTC  

---

## Key Observations
1. **All listed cycles show negative returns.**  
   - Each triangular loop produced a small **loss** rather than profit.  
   - Returns range from about **-0.03% to -0.09%**.  

2. **PnL impact is minor but consistent.**  
   - Example: CHF→EUR→USD→CHF lost **-0.0376%**, equal to **-3.58 CHF** on a 10,000 CHF start.  
   - EUR→USD→JPY→EUR lost **-0.0835%**, equal to **-8.35 EUR** on a 10,000 EUR start.  
   - AUD→JPY→USD→AUD lost **-0.09496%**, equal to **-9.50 AUD** on a 10,000 AUD start.  

3. **Why no arbitrage opportunities?**  
   - After factoring in spreads, fees, and slippage, cross-rates are **internally consistent**.  
   - The negative PnL represents the **transaction cost drag** inherent in trading triangles under realistic assumptions.  

---

## Conclusion
- With the given demo data, **no profitable triangular arbitrage exists**.  
- Instead, every cycle yields a small negative return due to fees and slippage.  
- In live markets, profitable opportunities may appear **only briefly**, usually in highly liquid pairs with momentary mispricings.  
- Next step: connect to **live FX feeds** and reduce `FEE_BPS` / `SLIPPAGE_BPS` to test sensitivity.  

---
